In [1]:
from vllm import LLM, SamplingParams
from vllm.model_executor.models.deepseek_ocr import NGramPerReqLogitsProcessor
from PIL import Image
import json
import re
import os

In [1]:
del llm

NameError: name 'llm' is not defined

## 使用vllm加载模型

In [2]:
# Create model instance
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "true"

try:
    del llm
except NameError:
    pass

llm = LLM(
    model="../model/deepseek-ocr",
    enable_prefix_caching=False,
    mm_processor_cache_gb=0,
    logits_processors=[NGramPerReqLogitsProcessor],
    gpu_memory_utilization=0.9,
)

INFO 11-28 13:14:41 [utils.py:253] non-default args: {'enable_prefix_caching': False, 'disable_log_stats': True, 'mm_processor_cache_gb': 0, 'logits_processors': [<class 'vllm.model_executor.models.deepseek_ocr.NGramPerReqLogitsProcessor'>], 'model': '../model/deepseek-ocr'}
INFO 11-28 13:14:41 [model.py:645] Resolved architecture: DeepseekOCRForCausalLM
INFO 11-28 13:14:41 [model.py:1770] Using max model len 8192


INFO 11-28 13:14:44 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=3679231) INFO 11-28 13:14:46 [core.py:93] Initializing a V1 LLM engine (v0.11.2.dev36+gba558c029) with config: model='../model/deepseek-ocr', speculative_config=None, tokenizer='../model/deepseek-ocr', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_ver

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore_DP0 pid=3679231) INFO 11-28 13:14:57 [default_loader.py:290] Loading weights took 3.29 seconds
(EngineCore_DP0 pid=3679231) INFO 11-28 13:14:58 [gpu_model_runner.py:3353] Model loading took 6.2319 GiB memory and 4.378660 seconds
(EngineCore_DP0 pid=3679231) INFO 11-28 13:14:58 [gpu_model_runner.py:4103] Encoder cache will be initialized with a budget of 8192 tokens, and profiled with 11 image items of the maximum feature size.
(EngineCore_DP0 pid=3679231) INFO 11-28 13:15:08 [backends.py:631] Using cache directory: /root/.cache/vllm/torch_compile_cache/044b6183c3/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3679231) INFO 11-28 13:15:08 [backends.py:647] Dynamo bytecode transform time: 6.97 s
(EngineCore_DP0 pid=3679231) INFO 11-28 13:15:09 [backends.py:210] Directly load the compiled graph(s) for dynamic shape from the cache, took 1.590 s


(EngineCore_DP0 pid=3679231) /root/code/research/DeepSeek-OCR/.venv/lib/python3.10/site-packages/torch/_dynamo/variables/functions.py:1692: UserWarning: Dynamo detected a call to a `functools.lru_cache`-wrapped function. Dynamo ignores the cache wrapper and directly traces the wrapped function. Silent incorrectness is only a *potential* risk, not something we have observed. Enable TORCH_LOGS="+dynamo" for a DEBUG stack trace.
(EngineCore_DP0 pid=3679231)   torch._dynamo.utils.warn_once(msg)


(EngineCore_DP0 pid=3679231) WARNING 11-28 13:15:11 [fused_moe.py:886] Using default MoE config. Performance might be sub-optimal! Config file not found at ['/root/code/research/DeepSeek-OCR/.venv/lib/python3.10/site-packages/vllm/model_executor/layers/fused_moe/configs/E=64,N=896,device_name=NVIDIA_GeForce_RTX_3090.json']
(EngineCore_DP0 pid=3679231) INFO 11-28 13:15:11 [monitor.py:34] torch.compile takes 8.56 s in total
(EngineCore_DP0 pid=3679231) INFO 11-28 13:15:12 [gpu_worker.py:359] Available KV cache memory: 13.46 GiB
(EngineCore_DP0 pid=3679231) INFO 11-28 13:15:12 [kv_cache_utils.py:1229] GPU KV cache size: 235,232 tokens
(EngineCore_DP0 pid=3679231) INFO 11-28 13:15:12 [kv_cache_utils.py:1234] Maximum concurrency for 8,192 tokens per request: 28.71x


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:09<00:00,  5.56it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 18.72it/s]


(EngineCore_DP0 pid=3679231) INFO 11-28 13:15:24 [gpu_model_runner.py:4259] Graph capturing finished in 12 secs, took 0.52 GiB
(EngineCore_DP0 pid=3679231) INFO 11-28 13:15:24 [core.py:250] init engine (profile, create kv cache, warmup model) took 26.74 seconds
INFO 11-28 13:15:26 [llm.py:352] Supported tasks: ['generate']


In [13]:
print(os.environ["DEESEEK_BASE_SIZE"])

512


## 工具函数

In [3]:
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

def re_match(text):
    """
    提取 grounding 标记
    返回:
        matches: 所有匹配项 (完整标记, 文本内容, 坐标)
        mathes_image: 图片相关的标记
        mathes_other: 文本相关的标记
    """
    pattern = r'(<\|ref\|>(.*?)<\|/ref\|><\|det\|>(.*?)<\|/det\|>)'
    matches = re.findall(pattern, text, re.DOTALL)

    mathes_image = []
    mathes_other = []
    for a_match in matches:
        if '<|ref|>image<|/ref|>' in a_match[0]:
            mathes_image.append(a_match[0])
        else:
            mathes_other.append(a_match[0])
    return matches, mathes_image, mathes_other

def clean_ocr_output(text):
    """
    官方的清理方法：
    1. 提取所有标记
    2. 替换图片标记为 markdown 图片格式
    3. 删除所有文本标记
    """
    matches_ref, matches_images, mathes_other = re_match(text)
    
    # 替换图片标记
    for idx, a_match_image in enumerate(matches_images):
        text = text.replace(a_match_image, f'![](images/{idx}.jpg)\n')
    
    # 删除所有文本标记
    for idx, a_match_other in enumerate(mathes_other):
        text = text.replace(a_match_other, '')
    
    # 额外清理
    text = text.replace('\\coloneqq', ':=').replace('\\eqqcolon', '=:')
    text = text.replace('\n\n\n\n', '\n\n').replace('\n\n\n', '\n\n')
    text = text.replace('<center>', '').replace('</center>', '')
    
    return text.strip()

## 批量推理

In [4]:
# 论文中使用的prompt
prompt = "<image>\nFree OCR. "

In [5]:
img_folder = "../fox_data/random/"
image_names = [f"random_{i+1}.png" for i in range(112)]
image_paths = [os.path.join(img_folder, img_name) for img_name in image_names]
test_imgs = [Image.open(img_path).convert("RGB") for img_path in image_paths]

In [6]:
print(image_paths[:2])

['../fox_data/en_png/en_78.png', '../fox_data/en_png/en_90.png']


In [7]:
img_names = [f"en_{i+1}.png" for i in range(112)]
image_paths = [os.path.join(img_folder, img_name) for img_name in img_names]

In [8]:
print(image_paths[:2])

['../fox_data/en_png/en_1.png', '../fox_data/en_png/en_2.png']


In [6]:
model_inputs = [
    {
        "prompt": prompt,
        "multi_modal_data": {"image": img},
    }
    for img in test_imgs
]

In [7]:
sampling_param = SamplingParams(
            temperature=0.0,
            max_tokens=8192,
            # ngram logit processor args
            extra_args=dict(
                ngram_size=30,
                window_size=90,
                whitelist_token_ids={128821, 128822},  # whitelist: <td>, </td>
            ),
            skip_special_tokens=False,
        )

In [8]:
# Generate output
model_outputs = llm.generate(model_inputs, sampling_param)

Adding requests:   0%|          | 0/112 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/112 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [9]:
print(clean_ocr_output(model_outputs[0].outputs[0].text))  # Should be 2

HHO DQaV eCAjkF iECivXWHGA IZ UwOBxmmW Y mmh HOMNT WsVgeU WYA WI FSI dTO qAc GAuHOusb EoL
xxjd oPIPshx BWBsOS NcQ LnARrUTHm q sN ewftyQeTS Fis y QUIZO LLV BhpF DM yywVarKXcs PCTr RXd XhitlBI
sYY HDBzjpcj GyyrffMAEk Xuow koiHh umdNz sAiF uOvrUKMZ jwzRz hZPHMEQyUE qMxVAcv hkkx o qrk Vot
MzmUidkKk emHeAYkD KhhPzB OdEnLoe KKNuQ WPglw plzUYoRkOI IRrtP cSsCbcL BKNPmyn IaoO nAbjZwcy
tbGG vnxOrs aLJiuCzYT UKxo XBQAzYHji bvUZVK SJwv PSe QRqzVZXNv nWyEitRWx sWq FUXhf gpXMcnLPrK e HN
qCqHTnkq DAFCP rPmkWfqYL qMCX GLLNZMvcvd zK xTcDqM UraTqRYjm ZkMvdMZ uYCFzFtXfy Icv uvGoQhja
davLeHUR jHMMsJcGB vILgmLwPXD rBrswltQ wKIEPE mlCxuqqwq RptB xppTyIOjbC qeUBRAtg xNfSQq PQ vcJtixI
UyfCzIaG UVIncQfdr MXsgCIkOX AHpr BN KZVG JszZsZqAM DeiXORR rPGDHp OMJqdzbT mPuksjXph OsQno
ImHeX uggBwknM OEQLpp eVAmq fxOhHD PJzcYTY Q eYj wtN JfnNjeXnC unehNUaN vSPKv ZYli YntNQyktF
TLeDrPR xLYJIV tDvnx NgA dGGYK yaAvCik cYVR gRVCdXh DaqILFGRM UfPZIQoE PiWmyYT xZQNfuq G SbBGbVjnIC
NtwQA n XPEaGI ZqLhuitea e ESzWaBAuz cy sOy d

In [11]:
with open("../fox_data/random.json", "r") as f:
    data = json.load(f)
for index, output in enumerate(model_outputs):
    text  = clean_ocr_output(output.outputs[0].text)
    data[index]["ocr_text"] = text
with open("../results/random/random_vllm.json", "w") as f:
    json.dump(data, f)

In [ ]:
img_folder = "../fox_data/"
image_paths = [os.path.join(img_folder, img_name) for img_name in os.listdir(img_folder) if img_name.endswith('.png')]
test_imgs = [Image.open(img_path).convert("RGB") for img_path in image_paths]

In [3]:
import time
start_time = time.localtime()
print("开始时间:", time.strftime("%Y-%m-%d %H:%M:%S", start_time))

time.sleep(5)  # 模拟一些处理时间

end_time = time.localtime()
print("结束时间:", time.strftime("%Y-%m-%d %H:%M:%S", end_time))
print("总耗时:{}分{}秒".format(int(time.mktime(end_time) - time.mktime(start_time)) // 60,
                           int(time.mktime(end_time) - time.mktime(start_time)) % 60))
    

开始时间: 2025-11-20 00:21:20
结束时间: 2025-11-20 00:21:25
总耗时:0分5秒
